In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()

print(f"Loaded {len(df_full)} games, {len(df_full.columns)} columns")

Loaded 2458 games, 95 columns


In [ ]:
conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()
df_full['home_win'] = (df_full['home_score'] > df_full['away_score']).astype(int)

feature_cols_v3 = [
    'home_recent_form', 'away_recent_form',
    'home_recent_point_diff', 'away_recent_point_diff',
    'home_qb_recent_yards', 'away_qb_recent_yards',
    'home_qb_recent_tds', 'away_qb_recent_tds',
    'home_qb_recent_ints', 'away_qb_recent_ints',
    'home_qb_recent_epa', 'away_qb_recent_epa',
    'home_rb_recent_rush_yards', 'away_rb_recent_rush_yards',
    'home_rb_recent_rush_epa', 'away_rb_recent_rush_epa',
    'home_rb_recent_rec_yards', 'away_rb_recent_rec_yards',
    'home_wrte_recent_rec_yards', 'away_wrte_recent_rec_yards',
    'home_wrte_recent_rec_epa', 'away_wrte_recent_rec_epa',
    'home_wrte_recent_targets', 'away_wrte_recent_targets',
    'home_qb_injury_flag', 'away_qb_injury_flag',
    'home_rb_injury_flag', 'away_rb_injury_flag',
    'home_wrte_injury_flag', 'away_wrte_injury_flag',
    'home_epa_allowed_recent', 'away_epa_allowed_recent',
    'home_yards_allowed_recent', 'away_yards_allowed_recent',
    'home_takeaways_recent', 'away_takeaways_recent',
    'home_coach_h2h_wins', 'h2h_games_played',
    'home_elo_pre', 'away_elo_pre',
    'div_game'
]

y = df_full['home_win']
X3 = df_full[feature_cols_v3].copy()
X3_imputed = pd.DataFrame(imputer.fit_transform(X3), columns=feature_cols_v3, index=X3.index)

train_mask = df_full['season'] <= 2021
test_mask = df_full['season'] >= 2022

X3_train, X3_test = X3_imputed[train_mask], X3_imputed[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

model.fit(X3_train, y_train)
elo_model_accuracy = model.score(X3_test, y_test)

print(f"With Elo (2022-2023 test): {elo_model_accuracy:.3f}")
print(f"Without Elo, same test set: 0.601")
print(f"Vegas, same test set: 0.663")

With Elo (2022-2023 test): 0.627
Without Elo, same test set: 0.601
Vegas, same test set: 0.663


In [ ]:
team_stats2 = nfl.load_team_stats(seasons=list(range(2015, 2024))).to_pandas()
team_stats2 = team_stats2[['game_id', 'season', 'week', 'team',
                            'passing_interceptions', 'fumbles_lost_total',
                            'def_interceptions', 'fumble_recovery_opp']]

team_stats2['giveaways'] = team_stats2['passing_interceptions'] + team_stats2['fumbles_lost_total']
team_stats2['takeaways'] = team_stats2['def_interceptions'] + team_stats2['fumble_recovery_opp']
team_stats2['turnover_margin'] = team_stats2['takeaways'] - team_stats2['giveaways']

team_stats2[['game_id', 'team', 'giveaways', 'takeaways', 'turnover_margin']].head(10)

,game_id,team,giveaways,takeaways,turnover_margin
0,2015_01_NO_ARI,ARI,1,1,0
1,2015_01_PHI_ATL,ATL,2,2,0
2,2015_01_BAL_DEN,BAL,2,1,-1
3,2015_01_IND_BUF,BUF,0,3,3
4,2015_01_CAR_JAX,CAR,1,3,2
5,2015_01_GB_CHI,CHI,1,0,-1
6,2015_01_CIN_OAK,CIN,0,2,2
7,2015_01_CLE_NYJ,CLE,5,1,-4
8,2015_01_NYG_DAL,DAL,3,0,-3
9,2015_01_BAL_DEN,DEN,1,2,1


In [ ]:
team_stats2 = team_stats2.sort_values(['team', 'season', 'week']).reset_index(drop=True)

for col in ['giveaways', 'turnover_margin']:
    team_stats2[f'recent_{col}'] = (
        team_stats2.groupby(['team', 'season'])[col]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    )

team_stats2[['team', 'season', 'week', 'giveaways', 'recent_giveaways', 'turnover_margin', 'recent_turnover_margin']].head(10)

,team,season,week,giveaways,recent_giveaways,turnover_margin,recent_turnover_margin
0,ARI,2015,1,1,NaN,0,NaN
1,ARI,2015,2,2,1.000000,0,0.0
2,ARI,2015,3,1,1.500000,3,0.0
3,ARI,2015,4,3,1.333333,-3,1.0
4,ARI,2015,5,0,1.750000,6,0.0
5,ARI,2015,6,3,1.400000,-3,1.2
6,ARI,2015,7,0,1.800000,2,0.6
7,ARI,2015,8,4,1.400000,-2,1.0
8,ARI,2015,10,3,2.000000,-2,0.0
9,ARI,2015,11,2,2.000000,-1,0.2


In [ ]:
turnover_recent = team_stats2[['team', 'game_id', 'recent_giveaways', 'recent_turnover_margin']]

home_to = turnover_recent.rename(columns={
    'team': 'home_team_std',
    'recent_giveaways': 'home_giveaways_recent',
    'recent_turnover_margin': 'home_turnover_margin_recent'
})
away_to = turnover_recent.rename(columns={
    'team': 'away_team_std',
    'recent_giveaways': 'away_giveaways_recent',
    'recent_turnover_margin': 'away_turnover_margin_recent'
})

df_full = df_full.drop(columns=[c for c in df_full.columns if 'giveaways_recent' in c or 'turnover_margin_recent' in c], errors='ignore')
df_full = df_full.merge(home_to, on=['home_team_std', 'game_id'], how='left')
df_full = df_full.merge(away_to, on=['away_team_std', 'game_id'], how='left')

df_full[df_full['game_id'].isin(['2019_05_CHI_OAK', '2016_03_SD_IND'])][
    ['game_id', 'home_team', 'away_team', 'home_turnover_margin_recent', 'away_turnover_margin_recent']]

,game_id,home_team,away_team,home_turnover_margin_recent,away_turnover_margin_recent
310,2016_03_SD_IND,IND,SD,-0.50,1.5
1137,2019_05_CHI_OAK,OAK,CHI,-0.25,1.5


In [ ]:
base_features = feature_cols_v3.copy()  # your current best feature set, with Elo

# Variant A: add turnover margin only
features_margin = base_features + ['home_turnover_margin_recent', 'away_turnover_margin_recent']

# Variant B: add giveaways only
features_giveaways = base_features + ['home_giveaways_recent', 'away_giveaways_recent']

# Variant C: add both
features_both = base_features + ['home_turnover_margin_recent', 'away_turnover_margin_recent',
                                   'home_giveaways_recent', 'away_giveaways_recent']

for name, feats in [('Baseline (no turnover feature)', base_features),
                     ('+ Turnover margin', features_margin),
                     ('+ Giveaways only', features_giveaways),
                     ('+ Both', features_both)]:
    X_temp = df_full[feats].copy()
    X_temp_imputed = pd.DataFrame(imputer.fit_transform(X_temp), columns=feats, index=X_temp.index)

    X_temp_train, X_temp_test = X_temp_imputed[train_mask], X_temp_imputed[test_mask]

    model.fit(X_temp_train, y_train)
    acc = model.score(X_temp_test, y_test)
    print(f"{name}: {acc:.3f}")

Baseline (no turnover feature): 0.627
+ Turnover margin: 0.629
+ Giveaways only: 0.629
+ Both: 0.629
